# Validação cruzada por local — 8 espécies

Pedido do professor: deixar cada local fora do treino uma vez, avaliar nele e
repetir para todos. Na checagem 3 (notebook 05) isso foi feito para lavali ×
resto (MCC 0,971 × 0,972 da divisão aleatória). Aqui, com as 8 espécies e a
configuração escolhida na Etapa D.

- dobras: locais com ≥ 100 gravações + "outros" (os menores juntos);
- avaliação: gravações do local nas partes de treino e validação da Etapa 2
  (teste intocado); treino: treino dos outros locais; checkpoint: validação
  dos outros locais;
- normalização e peso por classe recalculados só com o treino de cada dobra;
- configuração da Etapa D: 1 camada de 8 neurônios, Tanh, Adam 0,01, entropia
  cruzada com peso por classe, lote 64. 400 épocas (na Etapa D a melhor época
  nunca passou de 335), semente 0.

Comparação: acurácia balanceada 0,966 na divisão aleatória (Etapa D).

In [1]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

np.random.seed(42)
torch.manual_seed(42)

dados = np.load("../dados_processados/etapa2.npz", allow_pickle=True)
ctx = pd.read_csv("../dados_processados/contexto_simples.csv")
y_especie = dados["y_simples"]
assert (ctx["Espécie/Sonótipo"].to_numpy() == y_especie).all()

X_bruto = dados["X_simples"] * dados["desvio_simples"] + dados["media_simples"]  # desfaz a normalização
ordem_especies = pd.Series(y_especie).value_counts().index.tolist()
y = np.array([ordem_especies.index(e) for e in y_especie])
N = len(ordem_especies)

pos_treino, pos_val = dados["pos_treino_simples"], dados["pos_val_simples"]
pos_tv = np.concatenate([pos_treino, pos_val])
area = ctx["Área"].fillna("(vazio)").to_numpy()
tamanho = pd.Series(area[pos_tv]).value_counts()
grandes = tamanho[tamanho >= 100].index.tolist()
dobra = np.where(np.isin(area, grandes), area, "outros")
print(f"{len(grandes)} locais + 'outros' ({int(tamanho[tamanho < 100].sum())} gravações)")

20 locais + 'outros' (294 gravações)


In [2]:
def treina(Xtr, ytr, Xva, yva, n_epocas=400, semente=0):
    torch.manual_seed(semente)
    modelo = nn.Sequential(nn.Linear(5, 8), nn.Tanh(), nn.Linear(8, N))
    otim = torch.optim.Adam(modelo.parameters(), lr=0.01)
    n_c = np.maximum(np.bincount(ytr, minlength=N), 1)
    criterio = nn.CrossEntropyLoss(weight=torch.tensor(len(ytr) / (N * n_c), dtype=torch.float32))
    loader = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr)), batch_size=64, shuffle=True)
    tXva, tyva = torch.from_numpy(Xva), torch.from_numpy(yva)
    melhor, melhores = float("inf"), None
    for _ in range(n_epocas):
        modelo.train()
        for xb, yb in loader:
            otim.zero_grad()
            criterio(modelo(xb), yb).backward()
            otim.step()
        modelo.eval()
        with torch.no_grad():
            pv = criterio(modelo(tXva), tyva).item()
        if pv < melhor:
            melhor, melhores = pv, copy.deepcopy(modelo.state_dict())
    modelo.load_state_dict(melhores)
    return modelo


prev = np.full(len(y), -1)
for L in grandes + ["outros"]:
    tr = pos_treino[dobra[pos_treino] != L]
    va = pos_val[dobra[pos_val] != L]
    av = pos_tv[dobra[pos_tv] == L]
    media, desvio = X_bruto[tr].mean(axis=0), X_bruto[tr].std(axis=0)
    norm = lambda p: ((X_bruto[p] - media) / desvio).astype(np.float32)
    modelo = treina(norm(tr), y[tr], norm(va), y[va])
    modelo.eval()
    with torch.no_grad():
        prev[av] = modelo(torch.from_numpy(norm(av))).argmax(dim=1).numpy()
    print(f"  {L}: pronto")

In [3]:
def matriz_confusao(y_real, y_prev):
    m = np.zeros((N, N), dtype=int)
    for p, r in zip(y_prev, y_real):
        m[p, r] += 1  # linha = previsto, coluna = verdadeiro
    return m


def metricas(m):
    vp = np.diag(m).astype(float)
    sens = vp / m.sum(axis=0)
    pt = m.sum(axis=1)
    prec = np.divide(vp, pt, out=np.zeros_like(vp), where=pt > 0)
    f1 = np.divide(2 * prec * sens, prec + sens, out=np.zeros_like(vp), where=(prec + sens) > 0)
    return sens, prec, f1


m = matriz_confusao(y[pos_tv], prev[pos_tv])
sens, prec, f1 = metricas(m)
print(f"Por local, todas as dobras juntas (n = {len(pos_tv)}):")
print(f"  acurácia balanceada {sens.mean():.3f}   F1 macro {f1.mean():.3f}   acurácia {np.trace(m) / m.sum():.3f}")
print("  (divisão aleatória, Etapa D: acurácia balanceada 0,966)")

Por local, todas as dobras juntas (n = 11397):
  acurácia balanceada 0.955   F1 macro 0.915   acurácia 0.947
  (divisão aleatória, Etapa D: acurácia balanceada 0,966)


## Por espécie: de quantos locais ela depende?

A coluna "% no maior local" mostra quanto da espécie vem de um único local.
Quando esse local sai do treino, o modelo perde a maior parte dos exemplos.

In [4]:
linhas = []
for i, e in enumerate(ordem_especies):
    a = area[pos_tv][y[pos_tv] == i]
    cont = pd.Series(a).value_counts()
    linhas.append({"espécie": e, "n": len(a), "nº de locais": len(cont),
                   "maior local": cont.index[0], "% no maior local": round(100 * cont.iloc[0] / len(a)),
                   "sensibilidade por local": sens[i], "precisão por local": prec[i]})
pd.DataFrame(linhas).round(3)

In [5]:
linhas = []
for L in grandes + ["outros"]:
    p = pos_tv[dobra[pos_tv] == L]
    linhas.append({"local": L, "n": len(p), "nº de espécies": len(np.unique(y[p])),
                   "acurácia": (prev[p] == y[p]).mean()})
pd.DataFrame(linhas).sort_values("acurácia").round(3)